# Fase 4: Transfer Learning
## Comparación con ResNet50 preentrenada

Cargamos ResNet50 con pesos de ImageNet, congelamos las capas base y añadimos un clasificador para las 4 clases.

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

sys.path.append(os.path.abspath(".."))
from src.utils.config import (
    TRAINING_DIR, IMG_SIZE, BATCH_SIZE, EPOCHS, MODELS_DIR, CLASSES
)
from src.data.loader import load_image_paths, load_and_preprocess_image
from src.data.preprocessing import encode_labels, split_dataset
from src.data.augmentation import get_tf_generators
from src.evaluation.metrics import evaluate_model
from src.evaluation.plots import plot_training_history, plot_confusion_matrix

%matplotlib inline

In [ ]:
print("Cargando imágenes...")
train_paths, train_labels = load_image_paths(TRAINING_DIR)

X = []
y = []
for path, label in zip(train_paths, train_labels):
    img = load_and_preprocess_image(path, IMG_SIZE)
    if img is not None:
        X.append(img)
        y.append(label)

X = np.array(X)
y = encode_labels(np.array(y))
print(f"Imágenes cargadas: {X.shape}")

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = split_dataset(X, y)
train_ds, val_ds, test_ds = get_tf_generators(
    X_train, X_val, X_test, y_train, y_val, y_test
)

In [ ]:
base_model = tf.keras.applications.ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(*IMG_SIZE, 3)
)
base_model.trainable = False

model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(len(CLASSES), activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        os.path.join(MODELS_DIR, 'resnet50_finetuned.keras'),
        monitor='val_accuracy', save_best_only=True
    ),
    tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(patience=5, factor=0.5)
]

history = model.fit(
    train_ds, validation_data=val_ds,
    epochs=EPOCHS, callbacks=callbacks
)

In [ ]:
plot_training_history(history, '../docs/transfer_learning_curvas.png')

global_metrics, per_class, cm = evaluate_model(model, X_test, y_test)

print("MÉTRICAS TRANSFER LEARNING")
print(f"Accuracy:  {global_metrics['accuracy']:.4f}")
print(f"Precision: {global_metrics['precision']:.4f}")
print(f"Recall:    {global_metrics['recall']:.4f}")
print(f"F1-Score:  {global_metrics['f1_score']:.4f}")

plot_confusion_matrix(cm, '../docs/transfer_learning_confusion.png')

In [ ]:
model.save(os.path.join(MODELS_DIR, 'resnet50_finetuned.keras'))
print("Modelo guardado en models/resnet50_finetuned.keras")